# ScoutCard AI: run the tracker on a GPU

Runtime → Change runtime type → **T4 GPU** first. Then run the cells top to bottom.

Before this, on your Mac (in the repo folder), make the one file this notebook takes:

```bash
cd ~/scoutai && git pull origin claude/scoutcard-ai-v1-scaffold-c4bgpf
zip -r ~/Desktop/video-service.zip video-service \
  -x "video-service/.venv/*" "*/__pycache__/*" "video-service/tracks/*" \
     "video-service/detections/*" "video-service/debug/*" "video-service/recordings/*"
```

It holds the code, the cropped clips and each clip's calibration `.json`. The clips are game film, so use
your own Google account and delete the runtime when you're done (Runtime → Disconnect and delete runtime).

In [ ]:
!nvidia-smi -L
import torch
print("cuda:", torch.cuda.is_available())

## 1. Upload `video-service.zip` (drag it into the Files panel on the left, or run this cell)

In [ ]:
import os
if not os.path.exists("video-service.zip"):
    from google.colab import files
    files.upload()  # pick video-service.zip
!rm -rf video-service && unzip -q video-service.zip && ls video-service/clips

## 2. Install (the same OpenCV the Mac uses)

In [ ]:
!pip install -q "opencv-python-headless==5.0.0.93" transformers timm pillow pytest
import cv2; print(cv2.__version__)

## 3. Run the tracker, both ways, on plays 5 and 6
A = the whole frame cut into tiles (what ran before). B = `--region`: only the field, cut so a man is the same size to the model all play. Each run prints `time:` per step. Same model for both, so the comparison is fair.

In [ ]:
CHECKPOINT = "PekingU/rtdetr_v2_r50vd"
CONFIDENCE = "0.3"
SECONDS = 8
import os
os.environ["MODEL_CHECKPOINT"] = CHECKPOINT
os.environ["DETECT_CONFIDENCE"] = CONFIDENCE
%cd /content/video-service
!mkdir -p clips_two && cp clips/play_03.* clips/play_04.* clips_two/
print("=== A: whole-frame tiles ===")
!python -m eval.tracks clips_two/ --out tracks_tiles --seconds {SECONDS} --tiles 3 --detections detections_tiles/
print("=== B: the play's region, one man size ===")
!python -m eval.tracks clips_two/ --out tracks_region --seconds {SECONDS} --region --debug debug/ --detections detections_region/

## 4. Download the results
Send `results.zip` back (the detections hold numbers, no film; the pictures in `debug/` are of the region run). Also paste the printed `time:` lines.

In [ ]:
!zip -qr results.zip detections_tiles detections_region tracks_tiles tracks_region debug
from google.colab import files
files.download("results.zip")